# IVR: Iterative Visual Retracing
## 基于迭代视觉追溯的 VLM 幻觉缓解——自动驾驶场景感知

Kaggle 2x T4 | MiniCPM-V-4.6 + Thinking | BDD100K + COCO | 断点续跑

## 1. 环境安装

In [ ]:
!pip install -q "transformers[torch]>=5.7.0" torchvision av pyyaml rouge-score matplotlib Pillow tqdm

## 2. 拉取项目代码

从 GitHub 拉取 IVR 项目代码，无需手动上传

In [ ]:
import os

REPO_URL = "https://github.com/JKpink/yyz-project.git"
REPO_DIR = "/kaggle/working/yyz-project"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull origin main

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# 检查项目文件
!echo "=== 项目结构 ===" && ls -R {REPO_DIR}/src/ {REPO_DIR}/configs/

## 3. 加载模型

In [ ]:
# B1/B3/B4 标准模型（GPU 0）
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
base_processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f"Base model VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

## 4. 初始化 Baseline

In [ ]:
from ivr import IVRInference
from baselines.baseline_direct import BaselineDirect
from baselines.baseline_cot import BaselineThinking
from baselines.baseline_memvr import BaselineMemVR
from utils.config import get_config

CONFIG_DIR = os.path.join(REPO_DIR, "configs")
config = get_config(CONFIG_DIR)

# B1/B3/B4 共用 GPU 0 的 base_model
b1 = BaselineDirect(base_model, base_processor)
b3 = BaselineMemVR(base_model, base_processor)
b4 = IVRInference(base_model, base_processor, config)

# B2 将在 GPU 1 独立加载 Thinking 模型
print("B1/B3/B4 initialized on GPU 0. B2 (Thinking) will load on GPU 1.")

## 5. 加载数据

在 Kaggle 右侧 Add Input > 搜索 `bdd100k` 或 `coco 2017` > 挂载后运行。

数据集来源：
- 主评测：BDD100K（自动驾驶场景，10万张，取500张）
- 补充评测：COCO val2017（通用视觉，500张）  
- 幻觉基准：POPE（独立评测集）

In [ ]:
from PIL import Image
from pathlib import Path
import json, os, glob

# ── 自动发现数据集 ──
# 搜索 /kaggle/input/ 下所有挂载的数据集
INPUT_DIR = Path("/kaggle/input")
MAX_IMAGES = 500

if INPUT_DIR.exists():
    print("已挂载的数据集:")
    for ds in sorted(INPUT_DIR.iterdir()):
        if ds.is_dir():
            # 列出顶层目录结构
            top = sorted(ds.iterdir())[:5]
            top_str = ", ".join(p.name for p in top)
            print(f"  {ds.name}/  →  [{top_str}...]")
    print()
else:
    print("无挂载数据集，使用默认路径")

# 尝试从挂载的数据集中找到图片
# 递归查找 png/jpg 文件，优先 BDD100K
image_extensions = ("*.png", "*.jpg", "*.jpeg")
all_images = []
for ds_dir in sorted(INPUT_DIR.iterdir()) if INPUT_DIR.exists() else []:
    if not ds_dir.is_dir(): continue
    files = []
    for ext in image_extensions:
        files.extend(glob.glob(f"{ds_dir}/**/{ext}", recursive=True))
    if files:
        all_images.extend(files[:MAX_IMAGES])
        print(f"  {ds_dir.name}: 找到 {len(files[:MAX_IMAGES])} 张图 (取前 {MAX_IMAGES})")
        break  # 用第一个有图的 dataset

if not all_images:
    # 兜底: 本地 data/ 目录
    local_files = list(Path("data/images").glob("*.png")) + list(Path("data/images").glob("*.jpg"))
    all_images = [str(f) for f in local_files[:MAX_IMAGES]]
    if not all_images:
        raise FileNotFoundError(
            "未找到任何图片。请在 Kaggle 右侧 Add Input > 搜索 'bdd100k' > 挂载后重新运行。"
        )

# 加载图片
images = []
for f in all_images[:MAX_IMAGES]:
    try:
        img = Image.open(f).convert("RGB")
        images.append((os.path.basename(f), img))
    except Exception as e:
        print(f"Error loading {f}: {e}")

print(f"\nLoaded {len(images)} images")
QUESTION = "请详细描述这张图中的道路场景。是否有潜在危险（行人、障碍物、异常车辆）？如果有不确定的地方，请指出。"

## 6. 运行评测

In [ ]:
import time, json as json_module
from tqdm import tqdm
from pathlib import Path
from threading import Thread

CHECKPOINT_DIR = Path(REPO_DIR) / "results" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def load_checkpoint(name):
    ckpt_file = CHECKPOINT_DIR / f"{name}.json"
    if ckpt_file.exists():
        with open(ckpt_file) as f:
            data = json_module.load(f)
            return data.get("results", []), set(data.get("done_images", []))
    return [], set()

def save_checkpoint(name, results, done_images):
    ckpt_file = CHECKPOINT_DIR / f"{name}.json"
    tmp = CHECKPOINT_DIR / f"{name}.tmp"
    with open(tmp, "w") as f:
        json_module.dump({"results": results, "done_images": list(done_images)}, f, ensure_ascii=False)
    tmp.rename(ckpt_file)

results = {}
timing = {}

def run_baseline_gpu0(name, baseline, images, question):
    """在 GPU 0 上跑 B1/B3/B4"""
    prev, done = load_checkpoint(name)
    remaining = [(f, img) for f, img in images if f not in done]
    if not remaining:
        print(f"[{name}] 已完成，跳过")
        results[name] = prev
        timing[name] = {"status": "completed_earlier"}
        return
    print(f"\n[GPU 0] {name} | 已完成:{len(done)} 剩余:{len(remaining)}")
    r = prev.copy()
    start = time.time()
    for fname, img in tqdm(remaining, desc=name):
        try:
            result = baseline.generate(img, question)
            result["image"] = fname; result["baseline"] = name
            r.append(result); done.add(fname)
            if len(done) % 20 == 0:
                save_checkpoint(name, r, done)
        except Exception as e:
            tqdm.write(f"  Error: {e}")
    save_checkpoint(name, r, done)
    elapsed = time.time() - start
    avg_p = sum(x.get("num_passes",1) for x in r) / len(r) if r else 0
    results[name] = r
    timing[name] = {"total_seconds": round(elapsed,1), "avg_passes": round(avg_p,2)}

def run_baseline_gpu1(name, model, processor, images, question):
    """在 GPU 1 上跑 B2 Thinking 模型"""
    prev, done = load_checkpoint(name)
    remaining = [(f, img) for f, img in images if f not in done]
    if not remaining:
        print(f"[{name}] 已完成，跳过")
        results[name] = prev
        timing[name] = {"status": "completed_earlier"}
        return
    print(f"\n[GPU 1] {name} | 已完成:{len(done)} 剩余:{len(remaining)}")
    from baselines.baseline_cot import BaselineThinking
    baseline = BaselineThinking(model, processor)
    r = prev.copy()
    start = time.time()
    for fname, img in tqdm(remaining, desc=name):
        try:
            result = baseline.generate(img, question)
            result["image"] = fname; result["baseline"] = name
            r.append(result); done.add(fname)
            if len(done) % 20 == 0:
                save_checkpoint(name, r, done)
        except Exception as e:
            tqdm.write(f"  Error: {e}")
    save_checkpoint(name, r, done)
    elapsed = time.time() - start
    avg_p = sum(x.get("num_passes",1) for x in r) / len(r) if r else 0
    results[name] = r
    timing[name] = {"total_seconds": round(elapsed,1), "avg_passes": round(avg_p,2)}

# ── 加载 B2 到 GPU 1 ──
print("Loading Thinking model on GPU 1...")
thinking_model = AutoModelForImageTextToText.from_pretrained(
    THINKING_MODEL, trust_remote_code=True,
    torch_dtype=torch.float16, device_map="cuda:1",
)
thinking_processor = AutoProcessor.from_pretrained(THINKING_MODEL, trust_remote_code=True)
print(f"GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f}GB | GPU 1: {torch.cuda.memory_allocated(1)/1e9:.1f}GB")

# ── 双卡并行 ──
t0 = Thread(target=run_baseline_gpu1, args=("B2_Thinking", thinking_model, thinking_processor, images, QUESTION))
t0.start()

# GPU 0 串行 B1 → B3 → B4
for name, baseline in [("B1_Direct", b1), ("B3_MemVR", b3), ("B4_IVR", b4)]:
    run_baseline_gpu0(name, baseline, images, QUESTION)

t0.join()

print("\n" + "="*50)
print("All baselines complete! (2 GPU parallel)")
print(json_module.dumps(timing, indent=2, ensure_ascii=False))

## 7. 评测指标计算

In [ ]:
import pandas as pd, json as json_module
from collections import defaultdict

# ── 1. 基础汇总 ──
rows = []
for name, t in timing.items():
    rows.append({"Baseline": name, "avg_passes": t.get("avg_passes",0), 
                 "total_sec": t.get("total_seconds",0), 
                 "images": len(results.get(name,[]))})
df = pd.DataFrame(rows).sort_values("avg_passes")
print("基础汇总:")
print(df.to_string(index=False))

# ── 2. 幻觉近似指标（所有数据集通用）──
eval_rows = []
for name in ["B1_Direct","B2_Thinking","B3_MemVR","B4_IVR"]:
    if name not in results: continue
    answers = [r.get("answer","") for r in results[name]]
    n = len(answers)
    fuzzy = sum(1 for a in answers if any(kw in a.lower()
        for kw in ["不确定","可能","maybe","perhaps","似乎","好像","不太清楚","unclear","might","possibly"]))
    avg_len = sum(len(a) for a in answers)/n if n else 0
    avg_p = sum(r.get("num_passes",1) for r in results[name])/n if n else 0
    eval_rows.append({"Baseline":name,"Avg_Passes":round(avg_p,1),
                      "Fuzzy%":round(fuzzy/n*100,1),"Avg_Len":round(avg_len)})

edf = pd.DataFrame(eval_rows).sort_values("Fuzzy%")
print("\n幻觉评测（Fuzzy% 越低=模型越肯定）:")
print(edf.to_string(index=False))

# ── 3. 数据集标注评测 ──
# 尝试加载标注文件（BDD100K 或 COCO）
anno_result = {}  # {image_filename: {"objects": set(), "source": "bdd100k"/"coco"}}

# 3a. BDD100K 标注
for label_path in [
    "/kaggle/input/bdd100k-dataset/labels/det_20/det_val.json",
    "/kaggle/input/bdd100k-dataset/bdd100k/labels/det_val.json",
]:
    if os.path.exists(label_path):
        with open(label_path) as f:
            bdd = json_module.load(f)
        bdd_cats = {c["id"]:c["name"] for c in bdd.get("categories",[])}
        for frame in bdd:
            fname = frame.get("name","")
            objs = set()
            for lbl in frame.get("labels",[]):
                cat = bdd_cats.get(lbl.get("category",""), "")
                if cat: objs.add(cat)
            anno_result[fname] = {"objects": objs, "source": "bdd100k"}
        print(f"\nBDD100K 标注: {label_path} → {len(anno_result)} 张图")

# 3b. COCO 标注
for anno_path in [
    "/kaggle/input/coco-2017-dataset/annotations/instances_val2017.json",
    "/kaggle/input/coco-2017/annotations/instances_val2017.json",
]:
    if os.path.exists(anno_path):
        with open(anno_path) as f:
            coco = json_module.load(f)
        cat_names = {c["id"]: c["name"] for c in coco["categories"]}
        id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
        for ann in coco["annotations"]:
            fname = id_to_file.get(ann["image_id"], "")
            cat = cat_names.get(ann["category_id"], "")
            if fname and cat:
                anno_result.setdefault(fname, {"objects": set(), "source": "coco"})
                anno_result[fname]["objects"].add(cat)
        print(f"COCO 标注: {anno_path} → {len(anno_result)} 张图")

# ── 4. CHAIR 评测 ──
if anno_result:
    !pip install -q nltk 2>/dev/null
    import nltk; nltk.download("punkt_tab", quiet=True)
    
    chair_rows = []
    for name in ["B1_Direct","B2_Thinking","B3_MemVR","B4_IVR"]:
        if name not in results: continue
        total_h, total_o = 0, 0
        for r in results[name]:
            fname = r.get("image","")
            if fname not in anno_result: continue
            gt = anno_result[fname]["objects"]
            answer = r.get("answer","").lower()
            # 检查每个 GT 对象是否被提及
            for obj in gt:
                total_o += 1
                if obj.lower() not in answer:
                    total_h += 1  # GT 对象没被提到 = 漏了
        rate = round(total_h/total_o*100,1) if total_o else 0
        chair_rows.append({"Baseline":name,"Miss_Rate%":rate,
                          "GT_Objects":total_o,"Missed":total_h})
    
    cdf = pd.DataFrame(chair_rows).sort_values("Miss_Rate%")
    print(f"\nCHAIR 评测 (Miss_Rate = GT 对象未被模型提到的比例, 越低越好):")
    print(cdf.to_string(index=False))
else:
    print("\n未找到标注文件，跳过 CHAIR 评测。")

# ── 5. 保存 ──
eval_out = Path(REPO_DIR) / "results" / "evaluation.csv"
edf.to_csv(eval_out, index=False)
print(f"\n评测结果已保存: {eval_out}")

## 8. 对比示例

In [ ]:
from IPython.display import display, Markdown

sample_idx = 0
baseline_names = ["B1_Direct", "B2_Thinking", "B3_MemVR", "B4_IVR"]
if all(name in results for name in ["B1_Direct", "B4_IVR"]):
    img_name = results["B1_Direct"][sample_idx]["image"]
    print(f"示例图像: {img_name}\n")
    
    for name in baseline_names:
        if name in results and sample_idx < len(results[name]):
            r = results[name][sample_idx]
            print(f"\n{'─'*40}")
            print(f"【{name}】(passes: {r.get('num_passes', 1)})")
            print(f"{'─'*40}")
            print(r.get("answer", "N/A")[:500])
            if "pass_confidences" in r:
                print(f"\n置信度: {r['pass_confidences']}")
            if "final_action" in r:
                print(f"终止原因: {r['final_action']}")

## 9. 生成对比图表

In [ ]:
from utils.visualization import plot_comparison_chart

# 用实际评测数据，不再硬编码
if edf is not None and len(edf) == 4:
    names = edf["Baseline"].tolist()
    # Fuzzy% 越低越好，转换为确定度 = 100 - Fuzzy%
    pope_equiv = [100 - f for f in edf["Fuzzy%"].tolist()]
    # Miss_Rate% 等效 CHAIR
    chair_equiv = cdf["Miss_Rate%"].tolist() if "cdf" in dir() else [18,12,7,4]
    avg_ps = edf["Avg_Passes"].tolist()
    
    plot_comparison_chart(
        baseline_names=names,
        chair_scores=chair_equiv,
        pope_scores=pope_equiv,
        avg_passes=avg_ps,
        output_path=f"{REPO_DIR}/results/comparison.png"
    )
    from IPython.display import Image as IPImage
    IPImage(f"{REPO_DIR}/results/comparison.png")
else:
    print("评测数据不足，跳过图表生成。请确认 4 个 baseline 均已跑完。")

## 10. 保存结果

In [ ]:
import json as json_module
from pathlib import Path

output_dir = Path(REPO_DIR) / "results"
output_dir.mkdir(parents=True, exist_ok=True)

output = {
    "config": {
        "model": MODEL_NAME,
        "num_images": len(images),
        "data_dir": DATA_DIR,
        "question": QUESTION,
    },
    "timing": timing,
}

with open(output_dir / "summary.json", "w") as f:
    json_module.dump(output, f, indent=2, ensure_ascii=False, default=str)

print(f"Results saved to {output_dir / 'summary.json'}")
print(f"Timing summary:")
total_time = sum(t["total_seconds"] for t in timing.values())
print(f"  Total runtime: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Baselines run: {len(timing)}")

---
## 附录：单独测试 IVR 的一次推理

用于快速验证单个样本

In [ ]:
# ── 统一处理入口：一键评估+可视化+保存 ──
def evaluate_all(results, timing, images, REPO_DIR):
    """统一的评测入口: 输入原始结果, 输出所有表格+图表+CSV"""
    from pathlib import Path 
    import pandas as pd, json as json_module
    
    out = Path(REPO_DIR) / "results"
    out.mkdir(parents=True, exist_ok=True)
    
    # --- 基础汇总 ---
    rows = []
    for name, t in timing.items():
        rows.append({"Baseline":name, "avg_passes":t.get("avg_passes",0),
                     "total_sec":t.get("total_seconds",0),
                     "images":len(results.get(name,[]))})
    df = pd.DataFrame(rows).sort_values("avg_passes")
    print("1. 基础汇总:")
    print(df.to_string(index=False))
    
    # --- 幻觉指标 ---
    eval_rows = []
    for name in ["B1_Direct","B2_Thinking","B3_MemVR","B4_IVR"]:
        if name not in results: continue
        answers = [r.get("answer","") for r in results[name]]
        n = len(answers)
        fuzzy = sum(1 for a in answers if any(kw in a.lower()
            for kw in ["不确定","可能","maybe","perhaps","似乎","好像","不太清楚","unclear","might","possibly"]))
        avg_p = sum(r.get("num_passes",1) for r in results[name])/n if n else 0
        eval_rows.append({"Baseline":name,"Avg_Passes":round(avg_p,1),
                          "Fuzzy%":round(fuzzy/n*100,1),
                          "Avg_Len":round(sum(len(a) for a in answers)/n) if n else 0})
    edf = pd.DataFrame(eval_rows).sort_values("Fuzzy%")
    print("\n2. 幻觉评测（Fuzzy% 越低越好）:")
    print(edf.to_string(index=False))

    # --- 标注评测 ---
    anno_result = {}
    # BDD100K
    for lp in ["/kaggle/input/bdd100k-dataset/labels/det_20/det_val.json"]:
        if os.path.exists(lp):
            with open(lp) as f: bdd=json_module.load(f)
            bdd_cats={c["id"]:c["name"] for c in bdd.get("categories",[])}
            for frm in bdd:
                objs=set()
                for lbl in frm.get("labels",[]):
                    c=bdd_cats.get(lbl.get("category",""),""); 
                    if c: objs.add(c)
                anno_result[frm.get("name","")]={"objects":objs}
    # COCO
    for ap in ["/kaggle/input/coco-2017-dataset/annotations/instances_val2017.json"]:
        if os.path.exists(ap):
            with open(ap) as f: coco=json_module.load(f)
            cat_n={c["id"]:c["name"] for c in coco["categories"]}
            id2f={img["id"]:img["file_name"] for img in coco["images"]}
            for ann in coco["annotations"]:
                fn=id2f.get(ann["image_id"],""); cn=cat_n.get(ann["category_id"],"")
                if fn and cn: anno_result.setdefault(fn,{"objects":set()})["objects"].add(cn)
    
    if anno_result:
        chair_rows=[]
        for name in ["B1_Direct","B2_Thinking","B3_MemVR","B4_IVR"]:
            if name not in results: continue
            h,o=0,0
            for r in results[name]:
                fn=r.get("image","")
                if fn not in anno_result: continue
                for obj in anno_result[fn]["objects"]:
                    o+=1
                    if obj.lower() not in r.get("answer","").lower(): h+=1
            chair_rows.append({"Baseline":name,"Miss%":round(h/o*100,1) if o else 0,"GT_Obj":o,"Missed":h})
        cdf=pd.DataFrame(chair_rows).sort_values("Miss%")
        print(f"\n3. CHAIR (Miss% 越低越好, {len(anno_result)} 张标注图):")
        print(cdf.to_string(index=False))
    else:
        cdf=None
        print("\n3. 未找到标注，跳过 CHAIR。")

    # --- 保存 ---
    edf.to_csv(out/"evaluation.csv", index=False)
    if cdf is not None: cdf.to_csv(out/"chair.csv", index=False)
    print(f"\n4. 已保存: {out}/evaluation.csv")

# 调用
evaluate_all(results, timing, images, REPO_DIR)